# Notebook C — Evaluation: Zero-shot vs. SemSeg (environment classification)

Compares the two environment classifiers on a hand-labeled **multi-label** test set:

- **Accuracy:** macro **F1** + accuracy (subset / per-label) per method, and per-class F1.
- **Performance:** mean **runtime per frame** (from each notebook's run).

The better overall method is chosen for environment classification; per-class F1 also shows whether
a **combination** (pick the better method per class) would help.


## 1. Setup & paths

In [1]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import segmentation_common as sc

ENV_CLASSES = sc.CATEGORIES["environment"]
EVAL_DIR = Path("../dataset/eval")
TEST_IMAGES = EVAL_DIR / "images"
LABELS_CSV = EVAL_DIR / "env_labels.csv"                 # ground truth (multi-label)
PRED = {"zeroshot": EVAL_DIR / "env_pred_zeroshot.csv",  # from Notebook A
        "semseg":   EVAL_DIR / "env_pred_semseg.csv"}    # from Notebook B
RUNTIME = {"zeroshot": EVAL_DIR / "runtime_zeroshot.json",
           "semseg":   EVAL_DIR / "runtime_semseg.json"}
print("environment classes:", ENV_CLASSES)

environment classes: ['forest', 'open_field', 'water', 'industry', 'city']


## 2. Prepare the test-label template

The test set holds ~150 frames **per class** (multi-label: a frame may belong to several classes).
Labeling happens separately — run this once to scaffold an empty `env_labels.csv` from the images
in `dataset/eval/images/`, then fill each class column with 0/1.

In [2]:
def make_label_template(overwrite: bool = False) -> Path:
    imgs = sorted(TEST_IMAGES.glob("*.jpg")) + sorted(TEST_IMAGES.glob("*.png"))
    if not imgs:
        print("No images in", TEST_IMAGES, "- add test frames first.")
        return LABELS_CSV
    if LABELS_CSV.exists() and not overwrite:
        print(LABELS_CSV, "already exists (pass overwrite=True to regenerate).")
        return LABELS_CSV
    df = pd.DataFrame({"image": [p.name for p in imgs]})
    for c in ENV_CLASSES:
        df[c] = 0                                        # fill with 0/1 during labeling
    LABELS_CSV.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(LABELS_CSV, index=False)
    print(f"Wrote template for {len(df)} images -> {LABELS_CSV}")
    return LABELS_CSV


# make_label_template()   # uncomment to (re)generate the empty template

## 3. Load ground truth + predictions

Aligns each method's predictions to the labeled rows and returns 0/1 arrays over `ENV_CLASSES`.

In [3]:
def load_binary(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path).set_index("image")
    return df.reindex(columns=ENV_CLASSES).fillna(0).astype(int)


def load_eval():
    y_true = load_binary(LABELS_CSV)
    methods = {}
    for name, path in PRED.items():
        if path.exists():
            pred = load_binary(path)
            common = y_true.index.intersection(pred.index)   # only labeled & predicted images
            methods[name] = (y_true.loc[common].values, pred.loc[common].values, list(common))
    return methods

## 4. Accuracy metrics for both approaches  ←  the comparison cell

Computes **macro-F1**, subset accuracy, per-label accuracy and **per-class F1** for each method.
Falls back to a small synthetic demo if the labels/predictions don't exist yet.

In [4]:
def accuracy_table():
    rows, perclass = [], {}
    if LABELS_CSV.exists() and any(p.exists() for p in PRED.values()):
        methods = load_eval()
        source = "real data"
    else:
        # --- synthetic demo so the cell runs before data exists ---
        rng = np.random.default_rng(0)
        n, C = 60, len(ENV_CLASSES)
        yt = rng.integers(0, 2, (n, C))
        methods = {
            "zeroshot": (yt, (yt ^ (rng.random((n, C)) < 0.20)).astype(int), None),  # ~20% errors
            "semseg":   (yt, (yt ^ (rng.random((n, C)) < 0.12)).astype(int), None),  # ~12% errors
        }
        source = "SYNTHETIC DEMO (no env_labels.csv / predictions yet)"

    for name, (yt, yp, _) in methods.items():
        m = sc.multilabel_metrics(yt, yp)
        rows.append({"method": name, "n": len(yt),
                     "macro_F1": round(m["macro_f1"], 3),
                     "subset_accuracy": round(m["subset_accuracy"], 3),
                     "label_accuracy": round(m["label_accuracy"], 3)})
        perclass[name] = m["per_class_f1"]

    print("source:", source)
    summary = pd.DataFrame(rows)
    per_class_f1 = pd.DataFrame(perclass, index=ENV_CLASSES).round(3)
    return summary, per_class_f1


summary, per_class_f1 = accuracy_table()
print("\n== Overall ==");      display(summary)
print("== Per-class F1 =="); display(per_class_f1)

source: SYNTHETIC DEMO (no env_labels.csv / predictions yet)

== Overall ==


,method,n,macro_F1,subset_accuracy,label_accuracy
0,zeroshot,60,0.804,0.300,0.797
1,semseg,60,0.919,0.667,0.913


== Per-class F1 ==


,zeroshot,semseg
forest,0.829,0.919
open_field,0.806,0.900
water,0.781,0.918
industry,0.772,0.935
city,0.833,0.921


## 5. Performance: runtime per frame

In [5]:
rt_rows = []
for name, path in RUNTIME.items():
    if path.exists():
        d = json.load(open(path))
        rt_rows.append({"method": name, "ms_per_frame": round(d["ms_per_frame"], 1),
                        "fps": round(1000 / d["ms_per_frame"], 1), "n": d.get("n")})
runtime_table = pd.DataFrame(rt_rows) if rt_rows else pd.DataFrame(
    columns=["method", "ms_per_frame", "fps", "n"])
if rt_rows:
    display(runtime_table)
else:
    print("Run Notebooks A and B first to produce runtime_*.json")

,method,ms_per_frame,fps,n
0,zeroshot,26.5,37.8,14


## 6. Verdict & possible combination

- **Best overall method** = higher `macro_F1` at acceptable `ms_per_frame`.
- **Combination check:** if neither method has the best F1 on *every* class, a hybrid that takes
  each class from its stronger method could beat both — quantified below.

In [6]:
if per_class_f1.shape[1] == 2:
    best_per_class = per_class_f1.idxmax(axis=1)
    hybrid_f1 = per_class_f1.max(axis=1).mean()
    print("Best method per class:")
    print(best_per_class.to_string())
    print(f"\nMacro-F1 if combined (best per class): {hybrid_f1:.3f}")
    print("Per-method macro-F1:", {m: round(per_class_f1[m].mean(), 3) for m in per_class_f1.columns})
    print("\n-> A combination helps only if the per-class winner is split across methods.")
else:
    print("Need both methods' predictions to assess a combination.")

Best method per class:
forest        semseg
open_field    semseg
water         semseg
industry      semseg
city          semseg

Macro-F1 if combined (best per class): 0.919
Per-method macro-F1: {'zeroshot': np.float64(0.804), 'semseg': np.float64(0.919)}

-> A combination helps only if the per-class winner is split across methods.
